# RViT-D v2: Improved pipeline (MTCNN face crops + official DeeperForensics-1.0 code)

This notebook rebuilds the RViT-D training/evaluation pipeline with the data-quality and fine-tuning fixes that produced much stronger intra-dataset results:
- MTCNN face cropping instead of raw images
- A larger, leakage-checked FaceForensics++ training split
- Differential learning rate (backbone vs. classifier head)
- Validation-set threshold calibration (Youden's J)
- Early stopping, stochastic depth, label smoothing

The core extension — DeeperForensics-1.0's **actual released perturbation code** (`EndlessSora/DeeperForensics-1.0`, `perturbation/distortions.py`) applied to a Vision Transformer — is unchanged from your original notebook. Only the data pipeline and training recipe around it are improved.

Run cells top to bottom. Everything is saved to Google Drive under a new `RUN_NAME`, so it never overwrites your earlier runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU found. Runtime > Change runtime type > T4 GPU, then re-run.")


In [ ]:
!pip install -q -U timm opencv-python-headless
!pip install -q --no-deps facenet-pytorch
!pip install -q -U kagglehub


## 1. Clone DeeperForensics-1.0's official perturbation code
This is the actual released code from the paper (not a re-implementation).

In [ ]:
import numpy as np
import subprocess, sys
from pathlib import Path

REPO_DIR = Path('/content/DeeperForensics-1.0')

if not REPO_DIR.exists():
    print("Cloning EndlessSora/DeeperForensics-1.0 ...")
    r = subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/EndlessSora/DeeperForensics-1.0', str(REPO_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("Clone failed:\n" + r.stderr)
    print("Cloned.")
else:
    print("[skip] already cloned")

sys.path.insert(0, str(REPO_DIR / 'perturbation'))
import distortions as df_distortions

need = ['color_saturation', 'color_contrast', 'block_wise',
        'gaussian_noise_color', 'gaussian_blur', 'jpeg_compression']
missing = [n for n in need if not callable(getattr(df_distortions, n, None))]
assert not missing, f"Missing functions: {missing}"
print("DeeperForensics perturbations OK:", need)


## 2. Configuration

In [ ]:
import os, sys, json, random, shutil, subprocess
import numpy as np
from pathlib import Path

ROOT  = Path('/content/drive/MyDrive/RViT-D')
LOCAL = Path('/content/data')

RUN_NAME = 'run_v4_mtcnn_improved'   # new name: old checkpoints/results are untouched
RUN_DIR  = ROOT / 'runs' / RUN_NAME

for d in [ROOT, RUN_DIR, LOCAL]:
    d.mkdir(parents=True, exist_ok=True)

CFG = dict(
    model_name='vit_small_patch16_224',
    img_size=224,
    # data
    frames_per_real_video=8,     # face crops taken from each real video
    n_train_cap=4000,            # max images per class (train)
    n_val_cap=800,                # max images per class (val)
    n_test_cap=800,               # max images per class (intra test)
    n_cross=1500,                 # per class, cross-dataset tests
    # training
    epochs=10,
    patience=3,                   # early stopping: epochs without improvement
    batch_size=32,
    lr=2e-5,
    head_lr_mult=10,              # classifier head learns faster than the backbone
    weight_decay=0.05,
    drop_path=0.1,                # stochastic depth, reduces memorising
    label_smoothing=0.1,
    aug_prob=0.5,                 # probability of EACH DeeperForensics perturbation per image
    seed=42,
    num_workers=2,
)
json.dump(CFG, open(RUN_DIR / 'config.json', 'w'), indent=2)

FACES_DRIVE = ROOT / 'ff_faces'          # crops saved on Drive so a disconnect doesn't lose them
FACES_LOCAL = LOCAL / 'ffpp_faces'       # fast local copy used for training
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def list_images(d):
    return sorted(p for p in Path(d).rglob('*') if p.suffix.lower() in IMG_EXT)

def count_images(d):
    return sum(1 for p in Path(d).rglob('*') if p.suffix.lower() in IMG_EXT)

def find_test_dirs(root, names=('test',)):
    for dirpath, dirnames, _ in os.walk(root):
        if Path(dirpath).name.lower() in names:
            low = {d.lower(): d for d in dirnames}
            if 'real' in low and 'fake' in low:
                return Path(dirpath) / low['real'], Path(dirpath) / low['fake']
    raise FileNotFoundError(f"No matching real/fake folders under {root}")

set_seed(CFG['seed'])
print("Saving all results to:", RUN_DIR)


## 3. Download FaceForensics++ (c23) videos
~17 GB. Needs your Kaggle API token (same as before — `kaggle.json` or the new-style token).

In [ ]:
import kagglehub
from collections import Counter

kagglehub.login()   # a box appears: enter your Kaggle username and API token

free_gb = shutil.disk_usage('/content').free / 1e9
print(f"Free local disk: {free_gb:.0f} GB")
assert free_gb > 40, "Not enough free disk for an 18 GB download plus unzip."

path = kagglehub.dataset_download("xdxd003/ff-c23")
FF_RAW = Path(path)

while True:
    kids = list(FF_RAW.iterdir())
    if len(kids) == 1 and kids[0].is_dir():
        FF_RAW = kids[0]
    else:
        break
print("Dataset root:", FF_RAW)

for p in sorted(FF_RAW.iterdir()):
    if p.is_dir():
        files = [f for f in p.rglob('*') if f.is_file()]
        ext = Counter(f.suffix.lower() for f in files)
        print(f"{p.name:22s} {len(files):6d} files  {dict(ext)}")


## 4. Extract face crops with MTCNN
This is the key fix: tight, centered face crops instead of raw frames. Resumable — already-extracted videos are skipped, so re-running this cell after a disconnect is safe. Uses the four FaceForensics++ manipulation methods (Deepfakes, Face2Face, FaceSwap, NeuralTextures) for the fake class, matching your earlier protocol.

In [ ]:
import cv2, urllib.request
from tqdm.auto import tqdm
from facenet_pytorch import MTCNN

MAX_PAIRS = 250           # pairs per split to use; None = all (360/70/70 available)
METHODS = ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']
mtcnn = MTCNN(keep_all=False, device='cuda')

SPLIT_DIR = ROOT / 'splits'; SPLIT_DIR.mkdir(exist_ok=True)
base = "https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/"
splits = {}
for s in ['train', 'val', 'test']:
    fp = SPLIT_DIR / f"{s}.json"
    if not fp.exists():
        urllib.request.urlretrieve(base + f"{s}.json", fp)
    splits[s] = json.load(open(fp))
print({s: len(v) for s, v in splits.items()}, "pairs per split (expect 360 / 70 / 70)")

def extract_faces(video_path, out_dir, prefix, n_frames, margin=0.3):
    if list(out_dir.glob(f"{prefix}_*.jpg")):
        return -1  # already done
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return 0
    idxs = (np.linspace(0, total - 1, n_frames + 2, dtype=int)[1:-1]
            if n_frames > 1 else np.array([total // 2]))
    saved = 0
    for k, idx in enumerate(idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok: continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        boxes, probs = mtcnn.detect(rgb)
        if boxes is None: continue
        x1, y1, x2, y2 = boxes[int(np.argmax(probs))]
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        half = max(x2 - x1, y2 - y1) * (1 + margin) / 2
        H, W = rgb.shape[:2]
        a, b = int(max(cx - half, 0)), int(max(cy - half, 0))
        c, d = int(min(cx + half, W)), int(min(cy + half, H))
        if c - a < 32 or d - b < 32: continue
        out_dir.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out_dir / f"{prefix}_{k}.jpg"),
                    cv2.cvtColor(rgb[b:d, a:c], cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        saved += 1
    cap.release()
    return saved

F = CFG['frames_per_real_video']
new_crops, no_face = 0, []
for split, pairs in splits.items():
    if MAX_PAIRS: pairs = pairs[:MAX_PAIRS]
    real_ids = sorted({i for p in pairs for i in p})
    real_ids = [v for v in real_ids if (FF_RAW / 'original' / f"{v}.mp4").exists()]
    for vid in tqdm(real_ids, desc=f"{split} real"):
        r = extract_faces(FF_RAW / 'original' / f"{vid}.mp4",
                          FACES_DRIVE / split / 'real', f"real_{vid}", F)
        new_crops += max(r, 0)
        if r == 0: no_face.append(f"original/{vid}")
    fake_frames = max(1, round(F * len(real_ids) / max(1, len(METHODS) * len(real_ids))))
    for method in METHODS:
        for a, b in tqdm(pairs, desc=f"{split} {method}"):
            for vid_name in [f"{a}_{b}", f"{b}_{a}"]:
                vp = FF_RAW / method / f"{vid_name}.mp4"
                if not vp.exists(): continue
                r = extract_faces(vp, FACES_DRIVE / split / 'fake', f"{method}_{vid_name}", fake_frames)
                new_crops += max(r, 0)
                if r == 0: no_face.append(f"{method}/{vid_name}")

print(f"\nNew crops saved this run: {new_crops}")
print(f"Videos with no usable face: {len(no_face)}")
print("\nFace crops saved on Drive:")
for s in ['train', 'val', 'test']:
    print(f"  {s}  real={count_images(FACES_DRIVE/s/'real')}  fake={count_images(FACES_DRIVE/s/'fake')}")


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

assert FACES_DRIVE.exists(), "No face crops on Drive. Run the extraction cell above first."
shutil.copytree(FACES_DRIVE, FACES_LOCAL, dirs_exist_ok=True)

print("Face crops available:")
for s in ['train', 'val', 'test']:
    real = list_images(FACES_LOCAL / s / 'real')
    fake = list_images(FACES_LOCAL / s / 'fake')
    print(f"  {s:5s} real={len(real)}  fake={len(fake)}")

fig, ax = plt.subplots(2, 6, figsize=(16, 6))
for r, c in enumerate(['real', 'fake']):
    imgs = list_images(FACES_LOCAL / 'train' / c)
    files = random.Random(0).sample(imgs, min(6, len(imgs)))
    for a_, f in zip(ax[r], files):
        a_.imshow(Image.open(f)); a_.set_title(f"{c}\n{f.stem}", fontsize=8); a_.axis('off')
plt.tight_layout(); plt.show()


## 5. Build balanced splits (with a leakage check)
Makes sure no source video's frames appear in more than one split.

In [ ]:
def source_ids(path):
    stem = Path(path).stem.rsplit('_', 1)[0]
    return {p for p in stem.split('_') if p.isdigit()}

def video_name(path):
    return Path(path).stem.rsplit('_', 1)[0]

def build_split(root, split, n_cap, seed):
    real = list_images(Path(root) / split / 'real')
    fake = list_images(Path(root) / split / 'fake')
    n = min(len(real), len(fake), n_cap)
    rng = random.Random(seed)
    real, fake = rng.sample(real, n), rng.sample(fake, n)
    print(f"  {split:5s}: {n} per class | videos: real={len({video_name(p) for p in real})}, "
          f"fake={len({video_name(p) for p in fake})}")
    return real + fake, [0] * n + [1] * n          # 0 = real, 1 = fake

print("FF++ face crops:")
train_files, train_labels = build_split(FACES_LOCAL, 'train', CFG['n_train_cap'], CFG['seed'])
val_files,   val_labels   = build_split(FACES_LOCAL, 'val',   CFG['n_val_cap'],   CFG['seed'])
intra_files, intra_labels = build_split(FACES_LOCAL, 'test',  CFG['n_test_cap'],  CFG['seed'])

ids = {name: set().union(*(source_ids(f) for f in files)) if files else set()
       for name, files in [('train', train_files), ('val', val_files), ('test', intra_files)]}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap = ids[a] & ids[b]
    assert not overlap, f"Leakage between {a} and {b}: {sorted(overlap)[:10]}"
print("Leakage check passed: no source video appears in two splits.")
print(f"Totals: train={len(train_files)}, val={len(val_files)}, test={len(intra_files)}")


## 6. Cross-dataset test sets (OpenForensics, 140k Real and Fake Faces)
Same two cross-dataset sources you used before — unchanged.

In [ ]:
free_gb = shutil.disk_usage('/content').free / 1e9
print(f"Free local disk: {free_gb:.0f} GB")
assert free_gb > 10, "Low disk space. Delete /root/.cache/kagglehub/.../*.archive files or restart the runtime."

SOURCES = {
    'openforensics': 'manjilkarki/deepfake-and-real-images',
    'kaggle_140k':   'xhlulu/140k-real-and-fake-faces',
}

for folder, handle in SOURCES.items():
    link = LOCAL / folder
    if link.exists() or link.is_symlink():
        print(f"[skip] {folder} already set up")
        continue
    print(f"Downloading {handle} ...")
    path = kagglehub.dataset_download(handle)
    os.symlink(path, link)
    print(f"  -> {path}")

CROSS_NAMES = {
    'openforensics': 'Cross: OpenForensics',
    'kaggle_140k':   'Cross: 140k Real and Fake Faces',
}

cross_sets = {}
for folder, pretty in CROSS_NAMES.items():
    root = LOCAL / folder
    real_dir, fake_dir = find_test_dirs(root, names=('test',))
    real, fake = list_images(real_dir), list_images(fake_dir)
    n = min(len(real), len(fake), CFG['n_cross'])
    rng = random.Random(CFG['seed'])
    real, fake = rng.sample(real, n), rng.sample(fake, n)
    cross_sets[pretty] = (real + fake, [0] * n + [1] * n)
    print(f"{pretty}: {n} per class  (available: real={len(list_images(real_dir))}, fake={len(list_images(fake_dir))})")

train_val = set(train_files) | set(val_files)
for name, (files, _) in cross_sets.items():
    assert not (set(files) & train_val), f"{name} overlaps with training/validation files!"
print("Cross-dataset sets are separate from training and validation.")


## 7. DeeperForensics-1.0 perturbation augmentation + Dataset/DataLoader
Identical perturbation integration to your original notebook — uses the official code's exact functions and parameter tables.

In [ ]:
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

DF_PARAMS = {
    'CS':   [0.4, 0.3, 0.2, 0.1, 0.0],
    'CC':   [0.85, 0.725, 0.6, 0.475, 0.35],
    'BW':   [16, 32, 48, 64, 80],
    'GNC':  [0.001, 0.002, 0.005, 0.01, 0.05],
    'GB':   [7, 9, 13, 17, 21],
    'JPEG': [2, 3, 4, 5, 6],
}
DF_FUNCS = {
    'CS': df_distortions.color_saturation, 'CC': df_distortions.color_contrast,
    'BW': df_distortions.block_wise, 'GNC': df_distortions.gaussian_noise_color,
    'GB': df_distortions.gaussian_blur, 'JPEG': df_distortions.jpeg_compression,
}

def pil_to_bgr(img):
    return cv2.cvtColor(np.asarray(img.convert('RGB')), cv2.COLOR_RGB2BGR)

def bgr_to_pil(arr):
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB))

def apply_df_distortion(img, type_code, level):
    assert type_code in DF_PARAMS and 1 <= level <= 5
    return bgr_to_pil(DF_FUNCS[type_code](pil_to_bgr(img), DF_PARAMS[type_code][level - 1]))

def random_augment(img):
    for t in DF_PARAMS:
        if random.random() < CFG['aug_prob']:
            img = apply_df_distortion(img, t, random.randint(1, 5))
    return img

S = CFG['img_size']
M = (0.5, 0.5, 0.5)
eval_tf  = T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(M, M)])
train_tf = T.Compose([T.Resize((S, S)), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(M, M)])

class FaceDS(Dataset):
    def __init__(self, files, labels, pre_fn=None, train=False):
        assert len(files) == len(labels)
        self.files, self.labels, self.pre_fn = files, labels, pre_fn
        self.tf = train_tf if train else eval_tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        with Image.open(self.files[i]) as im:
            img = im.convert('RGB')
        if self.pre_fn is not None:
            img = self.pre_fn(img)
        return self.tf(img), torch.tensor(self.labels[i], dtype=torch.float32)

def make_loader(files, labels, pre_fn=None, train=False, shuffle=False, drop_last=False):
    return DataLoader(FaceDS(files, labels, pre_fn, train), batch_size=CFG['batch_size'],
                      shuffle=shuffle, num_workers=CFG['num_workers'], pin_memory=True,
                      drop_last=drop_last, persistent_workers=CFG['num_workers'] > 0)

# sanity check: original vs. the six perturbations at severity 4
im = Image.open(train_files[0]).convert('RGB')
names = ['original'] + list(DF_PARAMS)
imgs = [im] + [apply_df_distortion(im, t, 4) for t in DF_PARAMS]
fig, ax = plt.subplots(1, len(imgs), figsize=(18, 3))
for a_, n_, x in zip(ax, names, imgs):
    a_.imshow(x); a_.set_title(n_, fontsize=9); a_.axis('off')
plt.tight_layout(); plt.show()
print("Cell OK")


## 8. Model, training loop (differential LR, early stopping) and evaluation helpers

In [ ]:
import timm, torch.nn as nn, pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score, f1_score,
                             precision_score, recall_score, confusion_matrix)

device = torch.device('cuda')

def make_model():
    return timm.create_model(CFG['model_name'], pretrained=True, num_classes=1,
                             drop_path_rate=CFG['drop_path']).to(device)

@torch.no_grad()
def predict_logits(model, loader):
    model.eval(); ys, zs = [], []
    for x, y in loader:
        with torch.autocast('cuda', dtype=torch.float16):
            z = model(x.to(device, non_blocking=True)).squeeze(1)
        zs.append(z.float().cpu()); ys.append(y)
    return torch.cat(ys).numpy().astype(int), torch.cat(zs).numpy()

def best_threshold(y, z):
    # Youden's J statistic on raw logits (avoids probabilities saturating at 0/1)
    fpr, tpr, thr = roc_curve(y, z)
    i = int(np.argmax(tpr - fpr))
    return float(thr[i])

def summarize(y, z, thr):
    pred = (z >= thr).astype(int)
    tn, fp, fn, tp = [int(v) for v in confusion_matrix(y, pred, labels=[0, 1]).ravel()]
    return dict(auc=roc_auc_score(y, z), acc=accuracy_score(y, pred),
                bal_acc=0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1)),
                f1=f1_score(y, pred, zero_division=0),
                real_recall=tn / max(tn + fp, 1), fake_recall=tp / max(tp + fn, 1),
                tn=tn, fp=fp, fn=fn, tp=tp)

def train_model(name, use_aug):
    run = RUN_DIR / name; run.mkdir(parents=True, exist_ok=True)
    best_path, meta_path, done = run / 'best.pt', run / 'meta.json', run / 'done.flag'

    if done.exists() and best_path.exists() and meta_path.exists():
        print(f"[{name}] finished earlier, loading best checkpoint.")
        m = make_model(); m.load_state_dict(torch.load(best_path, map_location=device))
        return m, json.load(open(meta_path))['threshold']

    set_seed(CFG['seed'])
    tr = make_loader(train_files, train_labels, pre_fn=random_augment if use_aug else None,
                     train=True, shuffle=True, drop_last=True)
    va = make_loader(val_files, val_labels)

    model = make_model()
    head = list(model.get_classifier().parameters())
    head_ids = {id(p) for p in head}
    body = [p for p in model.parameters() if id(p) not in head_ids]
    opt = torch.optim.AdamW([{'params': body, 'lr': CFG['lr']},
                             {'params': head, 'lr': CFG['lr'] * CFG['head_lr_mult']}],
                            weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['epochs'])
    scaler = torch.amp.GradScaler('cuda')
    bce = nn.BCEWithLogitsLoss()
    eps = CFG['label_smoothing']

    best_score, best_epoch, bad = -1.0, 0, 0
    for epoch in range(1, CFG['epochs'] + 1):
        model.train(); tot = n = 0
        for x, y in tqdm(tr, desc=f"{name} epoch {epoch}/{CFG['epochs']}"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            y_smooth = y * (1 - eps) + 0.5 * eps
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                z = model(x).squeeze(1)
                loss = bce(z, y_smooth)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tot += loss.item() * x.size(0); n += x.size(0)
        sched.step()

        y_val, z_val = predict_logits(model, va)
        thr = best_threshold(y_val, z_val)
        r = summarize(y_val, z_val, thr)
        log = dict(epoch=epoch, train_loss=round(tot / n, 4), val_auc=round(r['auc'], 4),
                   val_bal_acc=round(r['bal_acc'], 4), val_thr=round(thr, 3))
        print(log)

        score = r['bal_acc']
        if score > best_score:
            best_score, best_epoch, bad = score, epoch, 0
            torch.save(model.state_dict(), best_path)
            json.dump(dict(threshold=thr, val_bal_acc=score, epoch=epoch), open(meta_path, 'w'))
            print(f"  saved best (epoch {epoch}, val balanced acc {score:.4f})")
        else:
            bad += 1
            if bad >= CFG['patience']:
                print(f"  early stopping at epoch {epoch} (no improvement for {CFG['patience']} epochs)")
                break

    done.touch()
    model.load_state_dict(torch.load(best_path, map_location=device))
    return model, json.load(open(meta_path))['threshold']


## 9. Train the plain ViT baseline (no DeeperForensics augmentation)

In [ ]:
vit_plain, thr_plain = train_model('vit_plain', use_aug=False)
print("val-tuned threshold (logit):", thr_plain)


## 10. Train RViT-D (with DeeperForensics-1.0 perturbation augmentation)

In [ ]:
rvitd, thr_rvitd = train_model('rvitd', use_aug=True)
print("val-tuned threshold (logit):", thr_rvitd)


## 11. Results: intra-dataset and cross-dataset evaluation

In [ ]:
models = {'ViT (no aug)': (vit_plain, thr_plain), 'RViT-D': (rvitd, thr_rvitd)}
test_sets = {'Intra: FF++ test': (intra_files, intra_labels), **cross_sets}

rows, store = [], {}
for mname, (m, thr) in models.items():
    for dname, (f, l) in test_sets.items():
        y, z = predict_logits(m, make_loader(f, l))
        store[(mname, dname)] = (y, z)
        r = summarize(y, z, thr)
        rows.append(dict(model=mname, test_set=dname, n=len(y), AUC=r['auc'], Accuracy=r['acc'],
                         BalancedAccuracy=r['bal_acc'], F1=r['f1'],
                         RealRecall=r['real_recall'], FakeRecall=r['fake_recall']))

results = pd.DataFrame(rows).round(4)
results.to_csv(RUN_DIR / 'results.csv', index=False)
print(results.to_string(index=False))


## 12. Score distributions (diagnostic)

In [ ]:
fig, axes = plt.subplots(len(models), len(test_sets), figsize=(5 * len(test_sets), 3.6 * len(models)), squeeze=False)
for r, mname in enumerate(models):
    for c, dname in enumerate(test_sets):
        y, z = store[(mname, dname)]
        ax = axes[r, c]
        ax.hist(z[y == 0], bins=40, alpha=0.6, label='real')
        ax.hist(z[y == 1], bins=40, alpha=0.6, label='fake')
        ax.axvline(0, color='k', ls='--', lw=1)
        ax.set_title(f"{mname}\n{dname}", fontsize=9); ax.set_xlabel('logit (higher = more fake)')
        if r == 0 and c == 0: ax.legend()
plt.tight_layout(); plt.savefig(RUN_DIR / 'score_hist.png', dpi=150); plt.show()


## 13. Robustness curves across DeeperForensics-1.0 perturbation severity levels

In [ ]:
sub = {}
for dname, (f, l) in test_sets.items():
    idx = random.Random(0).sample(range(len(f)), min(600, len(f)))
    sub[dname] = ([f[i] for i in idx], [l[i] for i in idx])

rrows = []
for t in DF_PARAMS:
    for lv in range(1, 6):
        for dname, (f, l) in sub.items():
            for mname, (m, _) in models.items():
                ld = make_loader(f, l, pre_fn=lambda im, t=t, lv=lv: apply_df_distortion(im, t, lv))
                y, z = predict_logits(m, ld)
                rrows.append(dict(degradation=t, level=lv, test_set=dname, model=mname, AUC=roc_auc_score(y, z)))
    print("done", t)

rob = pd.DataFrame(rrows); rob.to_csv(RUN_DIR / 'robustness.csv', index=False)

fig, axes = plt.subplots(1, len(DF_PARAMS), figsize=(22, 4), squeeze=False)
for c, t in enumerate(DF_PARAMS):
    for mname in models:
        s = rob[(rob.degradation == t) & (rob.model == mname)].groupby('level')['AUC'].mean()
        axes[0, c].plot(s.index, s.values, marker='o', label=mname)
    axes[0, c].set_title(t, fontsize=10); axes[0, c].set_xlabel('severity'); axes[0, c].set_ylabel('AUC')
    axes[0, c].grid(alpha=0.3)
    if c == 0: axes[0, c].legend(fontsize=8)
plt.tight_layout(); plt.savefig(RUN_DIR / 'robustness.png', dpi=150); plt.show()


## 14. Final Drive listing

In [ ]:
print("Run folder contents:", RUN_DIR)
for p in sorted(RUN_DIR.rglob('*')):
    if p.is_file():
        print(" ", p.relative_to(RUN_DIR))
